In [1]:
from Dataset.TemplateBuild import GraphDataset, GraphSnapshot, DatasetBuilder
from Dataset.utils import GraphTraverse

graph_snapshot = GraphSnapshot('./Dataset/result/ML_Dataset/graph_snapshot.pkl')

c:\Users\Yahya\Documents\POGNN_Complete\gnnEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading graph from ./Dataset/result/ML_Dataset/graph_snapshot.pkl...


In [2]:
ds = DatasetBuilder('./Dataset/result/', graph_snapshot=graph_snapshot)

In [ ]:
# ds.create_dataset(intake_folder_path='./Dataset/drive-download/tissue dataset/')

Querying mygene for 8136 entrez IDs...
Matched 4573 / 4601 graph nodes to table rows


In [3]:
from torch_geometric.loader import DataLoader
from Dataset.TemplateBuild import GraphDataset, GraphSnapshot, DatasetBuilder

train_dataset = GraphDataset(root='./Dataset/result/ML_Dataset', split='train')
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

graph_snapshot = GraphSnapshot(pickle_filepath='./Dataset/result/ML_Dataset/graph_snapshot.pkl')
print(graph_snapshot.graph.graph['edge_index'].shape)

gt = GraphTraverse(graph_snapshot)

Loading graph from ./Dataset/result/ML_Dataset/graph_snapshot.pkl...
torch.Size([2, 9706])


In [4]:
NUM_GENES = len(ds.gs.graph.nodes())
NUM_GENES

4601

In [5]:
tissue_descriptions = {0: 'Adipose Subcutaneous', 1: 'Artery Tibial', 2: 'Breast Mammary Tissue', 3: 'Cells Cultured Fibroblasts', 4: 'Esophagus Mucosa', 5: 'Lung', 6: 'Muscle Skeletal', 7: 'Nerve Tibial', 8: 'Thyroid', 9: 'Whole Blood'}

In [6]:
from Model.embeddings import BioBERTEmbeddings

biobert = BioBERTEmbeddings()
label_embeddings = biobert.get_embeddings(list(tissue_descriptions.values()))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21622.92it/s]
[transformers] BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import torch

label_embeddings = torch.tensor(label_embeddings)
label_embeddings.shape

torch.Size([10, 768])

In [8]:
from Model.train_utils import train
from Model.model import TissueClassificationPipeline
import wandb

EMB_DIM = 768

run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="yoyo458",
    # Set the wandb project where this run will be logged.
    project="my-awesome-project",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.01,
        "architecture": "GNN",
        "dataset": "real-dataset-normal-tissue-n10",
        "epochs": 20,
    },
)

model = TissueClassificationPipeline(
    in_channels=1, gat_hidden=64, emb_dim=EMB_DIM,
    gat_heads=4, dropout=0.1, temperature=0.07,
)
model.set_label_embeddings(label_embeddings, list(tissue_descriptions.values()))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

train(model, train_loader, optimizer, epochs=20, run=run)

wandb: Loading settings from C:\Users\Yahya\.config\wandb\settings
wandb: [wandb.login()] Loaded credentials for http://localhost:8080 from C:\Users\Yahya\_netrc.
wandb: Currently logged in as: yoyo458 to http://localhost:8080. Use `wandb login --relogin` to force relogin


=== Stage 1: Training backbone ===


KeyboardInterrupt: 

In [11]:
valid_dataset = GraphDataset(root='./Dataset/result/ML_Dataset', split='valid')
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=True)

In [13]:
for batch in valid_loader:
    print(batch.batch)
    break

tensor([ 0,  0,  0,  ..., 31, 31, 31])


In [3]:
for batch in train_loader:
    print(batch)
    edge_idx = gt.fetch_edge_index(batch, src_node=4744, dst_node=147700)
    src_idx, dst_idx = int(edge_idx[0]), int(edge_idx[1])
    pathways = gt.fetch_edge_shared_pathways(batch, src_idx, dst_idx)
    print(pathways)
    break

DataBatch(x=[147232], y=[32], edge_index=[2, 310592], pathway_index=[2, 348544], mask=[147232], batch=[147232], ptr=[33])
tensor([[4197, 4197, 1528, 1528],
        [ 288,  292,  292,  288]])
